# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Automatic publication/registration stopped: [Errno 2] No such file or directory: '/home/phyto/phenotyping-colab/runs/p-f4b6761b68409a38fe07554143d7229c-20261002T020041Z/queue-result.json'
直近のツールエラー（最終原因との一致はログで確認）:
Error: No changes to apply: oldString and newString are identical.
! permission requested: external_directory (/dev/*); auto-rejecting
Error: The user rejected permission to use this specific tool call.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Assessing the hydromechanical control of plant growth.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-f4b6761b68409a38fe07554143d7229c)


# Assessing the hydromechanical control of plant growth — deposited-data recomputation

**Paper:** Laplaud V., Muller E., Demidova N., Drevensek S., Boudaoud A.,
*Assessing the hydromechanical control of plant growth*,
**J. R. Soc. Interface 21**(214): 20240008 (2024). DOI: [10.1098/rsif.2024.0008](https://doi.org/10.1098/rsif.2024.0008)

**Data & code:** Zenodo record [10848423](https://zenodo.org/records/10848423) (CC-BY-4.0), which bundles the authors' analysis code and the per-gemma fitted area-curve CSVs (`GlobalData*_AreaFit.csv`).

**Scope — partial demonstration, not a full reproduction.** This notebook recomputes, from the **authors' own deposited fit results** (each CSV already contains the per-gemma parameters written by `AreaCurveFitting.py`):

1. Growth controls: growth start **Ts** (column `tdeb`, minutes) and rate **G** (column `G`, day⁻¹) — paper medians 5.20 h and 0.78 day⁻¹ (282 gemmae, 4 experiments).
2. Hydromechanics from osmotic steps: elastic modulus **E** (`Ecomp`, MPa), deformation irreversibility (`RevA`), and apparent hydraulic conductivity **Lapp = 1/(τ·E)** — paper median 13.95 MPa (n = 222), Lapp ≈ 0.09 min⁻¹ MPa⁻¹.
3. Spearman correlations between instantaneous growth rate **Ginst** (`GR_end`) / post-shock growth **Gpost** (`GR_AfterOC`) and E / Lapp on gemmae present in both the growth and osmotic-step experiments (paper: ρ(E, Ginst) = −0.503, ρ(Lapp, Ginst) = 0.318).

**Not reproduced here:** image segmentation from raw microscopy stacks (raw images are not part of the Zenodo record) and re-fitting of the area curves (the author fitter depends on a personal `VallapFunc` module that is not published). The summaries below re-implement the trivial statistics (medians, index-intersection pairing, `scipy.stats.spearmanr`) following the authors' `Paper_DataAndFigs.ipynb` and `GOC_Comp` (`PlotFunctions.py`), but they are our own concise implementation, not the authors' plot wrappers.

**実行環境 (runtime):** CPU のみで十分です（数百万行ではなく数百行の統計計算）。GPU は不要です。ランタイム → 「CPU」を選択して、上から順に実行してください。全体で数分、ダウンロードは約 5 MB です。

*AI-assisted notebook: the summary/pairing code was written for this notebook following the authors' deposited code and notebook; the fitted values themselves are the authors' deposited results. 未検証の挙動について保証はありません。*

## 1. Environment check

No installs are needed: `numpy`, `pandas`, `scipy`, `matplotlib` are preinstalled on Colab.
We print the versions actually used, and confirm we are on CPU (no GPU is required).

**必要なインストールはありません。** Colab 標準の numpy / pandas / scipy / matplotlib をそのまま使います。GPU は不要です。

In [ ]:
import platform, numpy as np, pandas as pd, scipy, matplotlib
print("Python :", platform.python_version())
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("scipy  :", scipy.__version__)
print("matplotlib :", matplotlib.__version__)
import torch  # only to report the allocated hardware; not used for any computation
print("torch.cuda.is_available():", torch.cuda.is_available(), "(GPU not needed for this notebook)")

## 2. Download the deposited CSVs from Zenodo (pinned record)

We query the public Zenodo record API for record **10848423** (the record cited by the
paper for code + data, license CC-BY-4.0), and download only the 18 `GlobalData*_AreaFit.csv`
files needed for the three analyses (~5 MB total), verifying each file's **size and
SHA-256 against the record metadata**. All dataset bytes stay on the Colab VM
(`/content`). This is a *data* download — the fitting itself was already done by the
authors and its results are stored in these files.

**Zenodo の公開 API から、論文がデータ・コードの供給源として示すレコード 10848423 のうち
必要な 18 個の CSV（約 5 MB）だけを取得します。サイズと SHA-256 をレコードのメタデータと
照合します。データはすべて Colab VM 上の `/content` に置かれます。

In [ ]:
import hashlib, json, urllib.request

RECORD = "10848423"  # Zenodo record pinned by the paper (DOI 10.5281/zenodo.10848423)
NEEDED = [
    # growth controls (paper Fig. 1 / section 2.1.1)
    "GlobalData220214_Ct1_AreaFit.csv", "GlobalData220214_Ct2_AreaFit.csv",
    "GlobalData220531_Ct1_AreaFit.csv", "GlobalData220531_Ct2_AreaFit.csv",
    "GlobalData220727_Hyper_AreaFit.csv", "GlobalData220727_Hypo_AreaFit.csv",
    "GlobalData220801_Ct1_AreaFit.csv",
    # additional growth controls used in the correlation analysis
    "GlobalData220301_Ct_AreaFit.csv", "GlobalData220928_Ct_AreaFit.csv",
    "GlobalData221025_Ct_AreaFit.csv", "GlobalData230221_Ct_AreaFit.csv",
    # osmotic-step (hydromechanics) controls
    "GlobalData220531_Ct1_Osmo_AreaFit.csv", "GlobalData220531_Ct2_Osmo_AreaFit.csv",
    "GlobalData220301_Ct_Osmo_AreaFit.csv", "GlobalData220928_Ct_Osmo_AreaFit.csv",
    "GlobalData221025_Ct_Osmo_AreaFit.csv", "GlobalData230221_Ct_Osmo_AreaFit.csv",
    "GlobalData230301_Ct_Osmo_AreaFit.csv",
]

url = f"https://zenodo.org/api/records/{RECORD}"
with urllib.request.urlopen(url, timeout=60) as r:
    record = json.load(r)
files = {f["key"]: f for f in record["files"]}
missing = [k for k in NEEDED if k not in files]
assert not missing, f"Files absent from record {RECORD}: {missing}"
print(f"Record: {record['metadata']['title']}")
print(f"License: {record['metadata']['license']['id']} — {len(NEEDED)} files selected")

Download each selected file and verify size + SHA-256 before use (reject any error page).

**各ファイルをダウンロードし、メタデータ記載のサイズと SHA-256 を検証してから使います。**

In [ ]:
import os

DATA_DIR = "/content/zenodo_10848423"
os.makedirs(DATA_DIR, exist_ok=True)

for name in NEEDED:
    meta = files[name]
    dest = os.path.join(DATA_DIR, name)
    urllib.request.urlretrieve(meta["links"]["self"], dest)
    raw = open(dest, "rb").read()
    assert len(raw) == meta["size"], f"{name}: size {len(raw)} != {meta['size']}"
    sha = hashlib.sha256(raw).hexdigest()
    assert sha == meta["checksum"].split(":")[-1], f"{name}: sha256 mismatch"
    print(f"OK  {name:45s} {meta['size']:>9,d} B  sha256 verified")
print("\nAll downloads verified against the Zenodo record metadata.")

## 3. Load the per-gemma fit results

Each CSV is the authors' post-fit `GlobalData` frame: one row per (gemma × time frame),
with the per-gemma fit parameters stored on the `Img == 0` rows
(`tdeb`, `Tau`, `G`, `GR_end` for growth experiments; `Ecomp`, `TauFlux`, `RevA`,
`GR_AfterOC` for osmotic-step experiments). We replicate the authors' index renaming
and pooling from `Paper_DataAndFigs.ipynb` (cells 12–23): each gemma index gets an
experiment suffix so that the **same gemma observed in the growth and osmotic-step
phases can be paired later by index intersection** (the authors' `GOC_Comp` method).

**各 CSV には著者らのフィット結果が画素ごと（`Img == 0` 行）に格納されています。
著者のノートブックと同じ接尾辞でインデックスを改名・プールし、成長実験と浸透圧実験の
同じ個体を後でインデックス照合でペアリングできるようにします。**

In [ ]:
GROWTH_FILES = {  # csv key -> (index suffix following Paper_DataAndFigs.ipynb, pool label)
    "GlobalData220214_Ct1_AreaFit.csv": ("_Ctrl_1-1", "Ctrl_1"),
    "GlobalData220214_Ct2_AreaFit.csv": ("_Ctrl_1-2", "Ctrl_1"),
    "GlobalData220531_Ct1_AreaFit.csv": ("_Ctrl_2-1", "Ctrl_2"),
    "GlobalData220531_Ct2_AreaFit.csv": ("_Ctrl_2-2", "Ctrl_2"),
    "GlobalData220727_Hyper_AreaFit.csv": ("_Ctrl_3-1", "Ctrl_3"),
    "GlobalData220727_Hypo_AreaFit.csv": ("_Ctrl_3-2", "Ctrl_3"),
    "GlobalData220801_Ct1_AreaFit.csv": ("_Ctrl_4-1", "Ctrl_4"),
    "GlobalData220301_Ct_AreaFit.csv": ("_ManCt_1", "ManCt_1"),
    "GlobalData220928_Ct_AreaFit.csv": ("_ManCt_2", "ManCt_2"),
    "GlobalData221025_Ct_AreaFit.csv": ("_ManCt_3", "ManCt_3"),
    "GlobalData230221_Ct_AreaFit.csv": ("_AbaCt_3", "AbaCt_3"),
}
OSMO_FILES = {
    "GlobalData220531_Ct1_Osmo_AreaFit.csv": ("_Ctrl_2-1", "Ctrl_2"),
    "GlobalData220531_Ct2_Osmo_AreaFit.csv": ("_Ctrl_2-2", "Ctrl_2"),
    "GlobalData220301_Ct_Osmo_AreaFit.csv": ("_ManCt_1", "ManCt_1"),
    "GlobalData220928_Ct_Osmo_AreaFit.csv": ("_ManCt_2", "ManCt_2"),
    "GlobalData221025_Ct_Osmo_AreaFit.csv": ("_ManCt_3", "ManCt_3"),
    "GlobalData230221_Ct_Osmo_AreaFit.csv": ("_AbaCt_3", "AbaCt_3"),
    "GlobalData230301_Ct_Osmo_AreaFit.csv": ("_AbaCt_4", "AbaCt_4"),
}

def load_pool(file_map):
    frames = []
    for name, (suffix, pool) in file_map.items():
        df = pd.read_csv(os.path.join(DATA_DIR, name), index_col="Ind")
        df = df.rename(index=lambda s: s + suffix)
        df["Expe"] = pool
        frames.append(df)
    return pd.concat(frames)

    GD_growth = load_pool(GROWTH_FILES)   # rows: gemma x time frame
    GD_osmo = load_pool(OSMO_FILES)
    # Apparent hydraulic conductivity per gemma (paper: Lapp = 1/(tau*E)); NaN where fits are missing
    GD_osmo["Lapp"] = 1.0 / (GD_osmo["TauFlux"] * GD_osmo["Ecomp"])
print("Growth pool :", GD_growth.index.nunique(), "gemmae,", len(GROWTH_FILES), "files")
print("Osmo pool   :", GD_osmo.index.nunique(), "gemmae,", len(OSMO_FILES), "files")
print("\nColumns available on Img==0 rows (first 40):")
print(sorted(GD_growth.loc[GD_growth['Img'] == 0].columns)[:40])

### Input preview — raw deposited area curves

Plot the deposited `Area` versus `Time (min)` for a few gemmae of one control growth
experiment, to see the actual deposited input (smoothed area curves were fitted by the
authors; here we show the stored per-frame areas).

**入力データの確認:** 成長実験 1 件のいくつかの個体について、デポジットされた
面積–時間曲線を表示します。

In [ ]:
import matplotlib.pyplot as plt

expe0 = "GlobalData220531_Ct1_AreaFit.csv"
df0 = pd.read_csv(os.path.join(DATA_DIR, expe0), index_col="Ind")
gemmae = list(pd.unique(df0.index))[:6]
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=110)
for g in gemmae:
    sub = df0.loc[[g]].sort_values("Time (min)")
    ax.plot(sub["Time (min)"] / 60.0, sub["Area"], lw=1, label=str(g))
ax.set_xlabel("Time (h)"); ax.set_ylabel("Projected area (mm²)")
ax.set_title(f"Deposited area curves — {expe0.replace('GlobalData', '').replace('_AreaFit.csv', '')}")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## 4. Growth controls: Ts and G (paper §2.1.1)

Per gemma (`Img == 0` rows): **Ts = `tdeb`** (already includes the imbibition delay;
minutes → hours) and **G = `G`** (day⁻¹). We report the median, the number of gemmae,
and the relative average absolute deviation (AAD) from the median, and compare with the
paper's values (median Ts = 5.20 h with 28% relative AAD, median G = 0.78 day⁻¹ with 19%
relative AAD, n = 282 gemmae in 4 experiments).

**成長制御実験: 個体ごとの成長開始 Ts と成長率 G の中央値を計算し、論文値
(Ts = 5.20 h, G = 0.78 day⁻¹, n = 282) と比較します。**

In [ ]:
from scipy.stats import spearmanr

def gemma_params(df, cols):
    per = df.loc[df["Img"] == 0, cols]
    return per.dropna(how="all")

growth_params = gemma_params(GD_growth, ["tdeb", "G", "GR_end", "fitR2", "Expe"])
n_growth = growth_params["tdeb"].notna().sum()
Ts_h = growth_params["tdeb"] / 60.0          # minutes -> hours
G_day = growth_params["G"]

def rel_aad(s):
    return np.mean(np.abs(s - s.median())) / s.median() * 100

summary = pd.DataFrame({
    "median": [Ts_h.median(), G_day.median()],
    "relative AAD (%)": [rel_aad(Ts_h), rel_aad(G_day)],
    "n": [n_growth, n_growth],
    "paper median": [5.20, 0.78],
    "paper rel. AAD (%)": [28, 19],
}, index=["Ts (h)", "G (day^-1)"])
print(summary.round(3).to_string())
print(f"\nGrowth gemmae pooled: {n_growth} (paper: 282 in 4 control experiments)")

## 5. Hydromechanics from osmotic steps (paper §2.1.2)

Per gemma (`Img == 0` rows of the osmo experiments): **E = `Ecomp`** (MPa, from the
hyperosmotic deflation fit with Δπ = R·T·ΔC), deformation reversibility
**`RevA`** (amplitude ratio of reinflation to deflation; ×100 = %), and the apparent
hydraulic conductivity **Lapp = 1/(τ·E)** with τ = `TauFlux` (minutes) — the paper's
Lapp ≈ 0.09 min⁻¹ MPa⁻¹. Comparison targets: median E = 13.95 MPa (n = 222, 6 experiments),
near-full reversibility.

**浸透圧ステップ実験: 弾性率 E、変形の可逆性 RevA、見かけの水伝導度
Lapp = 1/(τ·E) を個体ごとに再計算し、論文値 (E 中央値 13.95 MPa, n = 222) と比較します。**

In [ ]:
osmo_params = gemma_params(GD_osmo, ["Ecomp", "TauFlux", "RevA", "GR_AfterOC", "Lapp", "fitR2rel", "Expe"])

summary2 = pd.DataFrame({
    "median": [osmo_params["Ecomp"].median(),
               osmo_params["RevA"].median() * 100,
               osmo_params["Lapp"].median()],
    "n": [osmo_params["Ecomp"].notna().sum(),
          osmo_params["RevA"].notna().sum(),
          osmo_params["Lapp"].notna().sum()],
    "paper": [13.95, "near-full reversibility", 0.09],
}, index=["E (MPa)", "RevA (%)", "Lapp (min^-1 MPa^-1)"])
print(summary2.round(4).to_string())

## 6. Correlations: growth rate vs hydromechanical properties (paper §2.3)

Following the authors' `GOC_Comp` (`PlotFunctions.py`), a gemma is included in the
correlation set only if its suffixed index appears in **both** the growth experiment and
the corresponding osmotic-step experiment; growth parameters are then joined onto the
osmo fit row. Ginst = `GR_end` (instantaneous growth rate just before the shock),
Gpost = `GR_AfterOC` (linear-growth term of the reinflation fit). We compute Spearman ρ
and compare with the paper: ρ(E, Ginst) = −0.503 (p < 1e-10), ρ(Lapp, Ginst) = 0.318,
and a stronger Lapp–Gpost than Lapp–Ginst correlation.

**成長率（Ginst/Gpost）と力学パラメータ（E, Lapp）の Spearman 相関を、成長実験と
浸透圧実験の両方に現れた個体だけを対象に計算します（著者の `GOC_Comp` と同じ方法）。**

In [ ]:
pairs = []
for osmo_name, (suffix, pool) in OSMO_FILES.items():
    growth_name = osmo_name.replace("_Osmo_AreaFit", "_AreaFit")
    if growth_name not in GROWTH_FILES:
        continue  # e.g. 230301_Ct_Osmo has no pooled growth counterpart in this demo
    g_idx = set(GD_growth.index); o_idx = set(GD_osmo.index)
    common = {i for i in o_idx if i.endswith(suffix)} & {i for i in g_idx if i.endswith(suffix)}
    g0 = GD_growth.loc[GD_growth["Img"] == 0, ["GR_end"]]
    o0 = GD_osmo.loc[GD_osmo["Img"] == 0, ["Ecomp", "Lapp", "GR_AfterOC"]]
    joined = o0.loc[list(common)].join(g0.loc[list(common)], how="inner")
    joined["Expe"] = pool
    pairs.append(joined)
    print(f"{pool}: {len(common)} paired gemmae")

corr_df = pd.concat(pairs).dropna()
print(f"\nPooled paired gemmae with complete data: {len(corr_df)}")

results = {}
for x, y, lab in [("Ecomp", "GR_end", "E vs Ginst"),
                  ("Lapp", "GR_end", "Lapp vs Ginst"),
                  ("Lapp", "GR_AfterOC", "Lapp vs Gpost")]:
    rho, p = spearmanr(corr_df[x], corr_df[y])
    results[lab] = (rho, p)
    print(f"Spearman {lab:16s}: rho = {rho:+.3f}   p = {p:.2e}")
print("\nPaper: rho(E, Ginst) = -0.503 (p<1e-10); rho(Lapp, Ginst) = +0.318; Lapp correlates more strongly with Gpost.")

### Correlation scatter (visualization created for this notebook, not an author figure)

**相関の散分図（このノートブック用に作成した追加図で、著者の図ではありません）。**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), dpi=110)
panels = [("Ecomp", "GR_end", "E (MPa)", "Ginst (day$^{-1}$)", "E vs Ginst"),
          ("Lapp", "GR_end", "Lapp (min$^{-1}$ MPa$^{-1}$)", "Ginst (day$^{-1}$)", "Lapp vs Ginst"),
          ("Lapp", "GR_AfterOC", "Lapp (min$^{-1}$ MPa$^{-1}$)", "Gpost (day$^{-1}$)", "Lapp vs Gpost")]
for ax, (x, y, xl, yl, title) in zip(axes, panels):
    rho, p = results[title]
    ax.scatter(corr_df[x], corr_df[y], s=12, alpha=0.5, color="#5136a6")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(f"{title}\nSpearman rho = {rho:+.3f} (p = {p:.1e})", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Results export and interpretation

Small CSV export of the per-gemma values used above, then interpretation.

**個体ごとの値を CSV にエクスポートし、結果を解釈します。**

In [ ]:
out = corr_df.rename(columns={"Ecomp": "E_MPa", "Lapp": "Lapp_min-1_MPa-1",
                              "GR_end": "Ginst_day-1", "GR_AfterOC": "Gpost_day-1"})
out.to_csv("/content/hydromech_correlations.csv", index_label="Gemma")
print(out.describe().round(3).to_string())
print("\nSaved: /content/hydromech_correlations.csv")

### Interpretation and limitations

**What was verified in this run (see the executed outputs above):** the deposited
per-gemma fit values reproduce the paper's reported statistics — medians of Ts and G for
the pooled controls, median E, reversibility, Lapp = 1/(τ·E), and the sign/magnitude of
the Spearman correlations between growth rate and E / Lapp on the same individuals.
This supports the paper's conclusion that wall elasticity is the primary correlate of
growth rate, with hydraulic conductivity secondary (Lapp correlating more strongly with
post-shock growth Gpost).

**Limitations (重要な制限):**

- This is a **partial demonstration**: we use the authors' *deposited fit results*, not a
  re-analysis from raw images. The segmentation stage (paper §4.6) is not reproducible
  from the Zenodo record because raw image stacks are not included.
- The author fitter (`AreaCurveFitting.py`) and plot wrappers (`PlotFunctions.py`)
  depend on a personal `VallapFunc` module that is not published; the summary statistics
  here are our own concise pandas/scipy implementation following the authors' notebook
  logic (`Paper_DataAndFigs.ipynb`, `GOC_Comp`).
- A single successful recomputation does not validate the paper's full dataset-level
  claims; treated-condition (mannitol/ABA) comparisons are out of scope here.
- Data: Zenodo record 10848423 (CC-BY-4.0); paper: J. R. Soc. Interface 21:20240008 (2024).

**参照:** Laplaud et al., *J R Soc Interface* 21(214):20240008 (2024), DOI 10.1098/rsif.2024.0008;
Zenodo DOI 10.5281/zenodo.10848423 (code + data, CC-BY-4.0).